# XLKD-00: ExpansionNet v2 → S1 Cross-Vocabulary Logit-KD Audit

Đây là nhánh ablation độc lập theo yêu cầu thử nghiệm thêm Logit KD cho cặp
Teacher T1–Student S1. Teacher T1 hiện tại là **ExpansionNet v2**, không phải
CLIP cũ.

Contract chính thức đã xác định:

- Teacher vocabulary: 10.000 token;
- Student vocabulary: 10.080 token;
- raw token IDs không tương thích;
- direct KL trên hai vector logits gốc bị cấm.

Vì vậy, XLKD-00 **không huấn luyện** và chưa đọc logits. Notebook xây dựng ánh
xạ theo token string đã chuẩn hóa, kiểm tra toàn bộ 113.287 pseudo-caption và
chỉ authorize GPU probability-mass probe nếu structural gates đạt.

Tên phương pháp chính xác của nhánh này là:

> ExpansionNet v2 → S1 Cross-Vocabulary Mapped Logit KD

Không được gọi là direct Logit KD.

## Kaggle inputs

1. Phase 01 Formal Output.
2. Phase 03B ExpansionNet v2 Offline Preflight output.
3. Phase 03E ExpansionNet v2 Replacement Handoff output.
4. ExpansionNet v2 offline model package chứa `MODEL_SOURCE_MANIFEST.json`
   và `demo_coco_tokens.pickle`.

Runtime: CPU, Internet Off, không MS COCO images, Test luôn `SEALED`.


## Cell 1 — Source discovery, authorization và leakage boundary


In [1]:
import csv, hashlib, json, os, pickle, re, time, unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//xlkd00_expansionnet_s1_compatibility",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
REPORT = OUT / "reports"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, REPORT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
P1, P3B, P3E = map(read_json, [P1_PATH, P3B_PATH, P3E_PATH])
assert P1["status"] == "ready"
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete" and P3E["teacher"] == "ExpansionNet v2"
assert P3B["direct_logit_kd_allowed"] is False
assert P3B["sequence_kd_required"] is True
assert P3E["direct_logit_kd_allowed"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
for marker in [P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

model_root_override = os.environ.get("KLCN_EXPANSIONNET_OFFLINE_ROOT")
if model_root_override:
    MODEL_ROOT = Path(model_root_override).resolve()
    TOKEN_PICKLE = (MODEL_ROOT / "demo_coco_tokens.pickle").resolve()
else:
    # Kaggle mounts notebook outputs below /kaggle/input/notebooks but Model
    # inputs below /kaggle/input/models. Search the complete input tree for
    # the ExpansionNet vocabulary instead of restricting discovery to
    # NOTEBOOK_ROOT.
    candidates = sorted(INPUT_ROOT.rglob("demo_coco_tokens.pickle"))
    assert len(candidates) == 1, (
        "Cần đúng một demo_coco_tokens.pickle dưới "
        f"{INPUT_ROOT}; tìm thấy {len(candidates)}: {candidates}. "
        "Hãy gắn đúng một ExpansionNet v2 offline Model input."
    )
    TOKEN_PICKLE = candidates[0].resolve()
    MODEL_ROOT = TOKEN_PICKLE.parent
SOURCE_MANIFEST_PATH = MODEL_ROOT / "MODEL_SOURCE_MANIFEST.json"
assert TOKEN_PICKLE.is_file() and SOURCE_MANIFEST_PATH.is_file()

write_json(AUDIT / "xlkd00_cell01_sources.json", {
    "schema_version": "xlkd00_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "requested_method": "cross-vocabulary mapped Logit KD",
    "raw_direct_logit_kd_allowed": False,
    "training_started": False,
    "logit_payload_read": False,
    "dataset_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-00 CELL 1 — SOURCE AND BOUNDARY")
print("=" * 78)
print("Status                    : PASSED")
print("Teacher / Student         : ExpansionNet v2 / S1")
print("Raw direct Logit KD       : PROHIBITED")
print("Mapped Logit KD audit     : STARTED")
print("Training / logits / data  : False / False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 code cells")


XLKD-00 CELL 1 — SOURCE AND BOUNDARY
Status                    : PASSED
Teacher / Student         : ExpansionNet v2 / S1
Raw direct Logit KD       : PROHIBITED
Mapped Logit KD audit     : STARTED
Training / logits / data  : False / False / False
Test Set status           : SEALED
Progress                  : 1 / 6 code cells


## Cell 2 — Upstream manifests, vocabulary source và pseudo corpus


In [2]:
STARTED = time.perf_counter()

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

p3b_rows = verify_manifest(P3B_PATH, P3B)
p3e_rows = verify_manifest(P3E_PATH, P3E)
source_manifest = read_json(SOURCE_MANIFEST_PATH)
assert source_manifest["model_id"] == "expansionnet_v2"
vocab_reference = source_manifest["vocabulary"]
assert TOKEN_PICKLE.name == vocab_reference["relative_path"]
assert TOKEN_PICKLE.stat().st_size == int(vocab_reference["size_bytes"])
assert sha256(TOKEN_PICKLE) == vocab_reference["sha256"]
assert int(vocab_reference["vocabulary_size"]) == 10000

compatibility_path = resolve_inside(
    P3B_PATH.parent,
    P3B["compatibility_contract"]["relative_path"],
)
assert sha256(compatibility_path) == (
    P3B["compatibility_contract"]["sha256"]
)
COMPATIBILITY = read_json(compatibility_path)
assert COMPATIBILITY["teacher_vocabulary_size"] == 10000
assert COMPATIBILITY["student_vocabulary_size"] == 10080
assert COMPATIBILITY["token_spaces_compatible"] is False

STUDENT_VOCAB_PATH = (
    P1_PATH.parent / "vocabulary" / "word_vocabulary_minfreq5.json"
).resolve()
TOKENIZER_PATH = (
    P1_PATH.parent / "vocabulary" / "tokenizer_contract.json"
).resolve()
assert STUDENT_VOCAB_PATH.is_file() and TOKENIZER_PATH.is_file()
STUDENT_VOCAB = read_json(STUDENT_VOCAB_PATH)
TOKENIZER = read_json(TOKENIZER_PATH)
assert STUDENT_VOCAB["vocabulary_size"] == 10080
assert TOKENIZER["vocabulary_size"] == 10080

PSEUDO_PATH = (
    P3E_PATH.parent
    / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent
    / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
assert PSEUDO_PATH.is_file() and PSEUDO_SUMMARY_PATH.is_file()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert PSEUDO_SUMMARY["unknown_tokens"] == 0
assert P3E["pseudo_caption_records"] == 113287

write_json(AUDIT / "xlkd00_cell02_integrity.json", {
    "schema_version": "xlkd00_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "upstream_manifest_rows_verified": len(p3b_rows) + len(p3e_rows),
    "teacher_vocabulary_sha256": sha256(TOKEN_PICKLE),
    "student_vocabulary_sha256": sha256(STUDENT_VOCAB_PATH),
    "pseudo_caption_rows_expected": 113287,
    "pseudo_caption_payload_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-00 CELL 2 — INTEGRITY")
print("=" * 78)
print("Upstream manifest rows    :", len(p3b_rows) + len(p3e_rows))
print("Teacher vocabulary        : 10,000 VERIFIED")
print("Student vocabulary        : 10,080 VERIFIED")
print("Pseudo-caption corpus     : 113,287 FOUND")
print("Pseudo payload read       : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 code cells")


XLKD-00 CELL 2 — INTEGRITY
Upstream manifest rows    : 27
Teacher vocabulary        : 10,000 VERIFIED
Student vocabulary        : 10,080 VERIFIED
Pseudo-caption corpus     : 113,287 FOUND
Pseudo payload read       : False
Test Set status           : SEALED
Progress                  : 2 / 6 code cells


## Cell 3 — Xây vocabulary map theo tokenizer Phase 1


In [3]:
STARTED = time.perf_counter()
# Pickle is loaded only after size and SHA-256 verification in Cell 2.
with TOKEN_PICKLE.open("rb") as handle:
    teacher_tokens = pickle.load(handle)
teacher_id_to_token = teacher_tokens["idx2word_list"]
teacher_word_to_id = teacher_tokens["word2idx_dict"]
student_id_to_token = STUDENT_VOCAB["id_to_token"]
student_token_to_id = {
    token: index for index, token in enumerate(student_id_to_token)
}
assert len(teacher_id_to_token) == 10000
assert len(teacher_word_to_id) == 10000
assert len(student_id_to_token) == 10080

token_pattern = re.compile(TOKENIZER["token_pattern"])
apostrophe_variants = {"’": "'", "‘": "'", "`": "'"}

def normalize_single_token(value):
    text = unicodedata.normalize("NFKC", str(value)).lower()
    for source, target in apostrophe_variants.items():
        text = text.replace(source, target)
    tokens = token_pattern.findall(text)
    return tokens[0] if len(tokens) == 1 else None

teacher_special = COMPATIBILITY["teacher_special_ids"]
student_special = COMPATIBILITY["student_special_ids"]
special_map = {
    int(teacher_special["pad"]): int(student_special["pad"]),
    int(teacher_special["bos"]): int(student_special["bos"]),
    int(teacher_special["eos"]): int(student_special["eos"]),
    int(teacher_special["unk"]): int(student_special["unk"]),
}
mapping = dict(special_map)
mapping_rows = []
for teacher_id, teacher_token in enumerate(teacher_id_to_token):
    if teacher_id in special_map:
        student_id = special_map[teacher_id]
        mapping_type = "special_semantic"
        normalized = None
    else:
        normalized = normalize_single_token(teacher_token)
        student_id = (
            student_token_to_id.get(normalized)
            if normalized is not None else None
        )
        mapping_type = (
            "normalized_exact"
            if student_id is not None else "unmapped"
        )
        if student_id is not None:
            mapping[teacher_id] = int(student_id)
    mapping_rows.append({
        "teacher_id": teacher_id,
        "teacher_token": teacher_token,
        "normalized_token": normalized,
        "student_id": student_id,
        "student_token": (
            student_id_to_token[student_id]
            if student_id is not None else None
        ),
        "mapping_type": mapping_type,
    })

mapped_count = len(mapping)
unmapped_count = 10000 - mapped_count
student_targets = list(mapping.values())
collision_count = len(student_targets) - len(set(student_targets))
vocabulary_count_coverage = mapped_count / 10000
assert mapped_count + unmapped_count == 10000
assert len(special_map) == 4

mapping_csv = RESULT / "xlkd00_teacher_to_student_vocabulary_map.csv"
with mapping_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(mapping_rows[0]))
    writer.writeheader()
    writer.writerows(mapping_rows)
MAPPING_PATH = RESULT / "xlkd00_teacher_to_student_vocabulary_map.json"
write_json(MAPPING_PATH, {
    "schema_version": "xlkd00_vocabulary_map_v1",
    "created_at_utc": now(),
    "teacher_vocabulary_size": 10000,
    "student_vocabulary_size": 10080,
    "mapped_teacher_tokens": mapped_count,
    "unmapped_teacher_tokens": unmapped_count,
    "vocabulary_count_coverage": vocabulary_count_coverage,
    "mapping_collisions": collision_count,
    "special_map": {
        str(source): target for source, target in special_map.items()
    },
    "mapping": {
        str(source): target for source, target in sorted(mapping.items())
    },
    "normalization": {
        "unicode": "NFKC",
        "lowercase": True,
        "token_pattern": TOKENIZER["token_pattern"],
    },
    "teacher_vocabulary_sha256": sha256(TOKEN_PICKLE),
    "student_vocabulary_sha256": sha256(STUDENT_VOCAB_PATH),
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd00_cell03_vocabulary_map.json", {
    "schema_version": "xlkd00_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "mapped_teacher_tokens": mapped_count,
    "unmapped_teacher_tokens": unmapped_count,
    "vocabulary_count_coverage": vocabulary_count_coverage,
    "mapping_collisions": collision_count,
    "special_tokens_mapped": "4 / 4",
    "raw_token_spaces_compatible": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-00 CELL 3 — VOCABULARY MAP")
print("=" * 78)
print("Mapped Teacher tokens     :", mapped_count, "/ 10000")
print("Unmapped Teacher tokens   :", unmapped_count)
print("Count coverage            :", f"{vocabulary_count_coverage:.4%}")
print("Mapping collisions        :", collision_count)
print("Special tokens            : 4 / 4 MAPPED")
print("Raw direct Logit KD       : STILL PROHIBITED")
print("Progress                  : 3 / 6 code cells")


XLKD-00 CELL 3 — VOCABULARY MAP
Mapped Teacher tokens     : 9959 / 10000
Unmapped Teacher tokens   : 41
Count coverage            : 99.5900%
Mapping collisions        : 0
Special tokens            : 4 / 4 MAPPED
Raw direct Logit KD       : STILL PROHIBITED
Progress                  : 3 / 6 code cells


## Cell 4 — Full pseudo-caption positional and occurrence audit


In [4]:
STARTED = time.perf_counter()
row_count = 0
total_teacher_occurrences = 0
mapped_occurrences = 0
unmapped_occurrences = Counter()
teacher_occurrences = Counter()
length_mismatch_rows = 0
position_mismatch_rows = 0
duplicate_image_ids = 0
seen_image_ids = set()

with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        row_count += 1
        image_id = int(record["image_id"])
        if image_id in seen_image_ids:
            duplicate_image_ids += 1
        seen_image_ids.add(image_id)
        teacher_ids = [int(value) for value in record["teacher_token_ids"]]
        student_ids = [int(value) for value in record["student_token_ids"]]
        if len(teacher_ids) != len(student_ids):
            length_mismatch_rows += 1
        mapped_sequence = []
        sequence_fully_mappable = True
        for teacher_id in teacher_ids:
            total_teacher_occurrences += 1
            teacher_occurrences[teacher_id] += 1
            if teacher_id in mapping:
                mapped_occurrences += 1
                mapped_sequence.append(mapping[teacher_id])
            else:
                sequence_fully_mappable = False
                unmapped_occurrences[teacher_id] += 1
        if (
            not sequence_fully_mappable
            or mapped_sequence != student_ids
        ):
            position_mismatch_rows += 1

assert row_count == 113287
occurrence_coverage = (
    mapped_occurrences / total_teacher_occurrences
)
length_alignment_rate = 1.0 - length_mismatch_rows / row_count
position_alignment_rate = 1.0 - position_mismatch_rows / row_count
unique_teacher_tokens_observed = len(teacher_occurrences)
unmapped_observed_tokens = len(unmapped_occurrences)

coverage_payload = {
    "schema_version": "xlkd00_pseudo_coverage_v1",
    "created_at_utc": now(),
    "rows": row_count,
    "unique_image_ids": len(seen_image_ids),
    "duplicate_image_ids": duplicate_image_ids,
    "teacher_token_occurrences": total_teacher_occurrences,
    "mapped_occurrences": mapped_occurrences,
    "unmapped_occurrences": (
        total_teacher_occurrences - mapped_occurrences
    ),
    "occurrence_weighted_coverage": occurrence_coverage,
    "unique_teacher_tokens_observed": unique_teacher_tokens_observed,
    "unmapped_observed_tokens": unmapped_observed_tokens,
    "length_mismatch_rows": length_mismatch_rows,
    "length_alignment_rate": length_alignment_rate,
    "position_mismatch_rows": position_mismatch_rows,
    "position_alignment_rate": position_alignment_rate,
    "scope": "Train+Restval pseudo-caption top-1 sequences only",
    "does_not_measure_soft_probability_mass": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}
COVERAGE_PATH = RESULT / "xlkd00_pseudo_caption_coverage.json"
write_json(COVERAGE_PATH, coverage_payload)
write_json(AUDIT / "xlkd00_cell04_pseudo_alignment.json", {
    "schema_version": "xlkd00_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "rows_verified": row_count,
    "occurrence_weighted_coverage": occurrence_coverage,
    "length_alignment_rate": length_alignment_rate,
    "position_alignment_rate": position_alignment_rate,
    "unmapped_observed_tokens": unmapped_observed_tokens,
    "soft_probability_mass_verified": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-00 CELL 4 — PSEUDO-CAPTION ALIGNMENT")
print("=" * 78)
print("Rows / unique images      :", row_count, "/", len(seen_image_ids))
print("Teacher token occurrences :", total_teacher_occurrences)
print("Occurrence coverage       :", f"{occurrence_coverage:.6%}")
print("Length alignment          :", f"{length_alignment_rate:.6%}")
print("Position exact alignment  :", f"{position_alignment_rate:.6%}")
print("Unmapped observed tokens  :", unmapped_observed_tokens)
print("Soft probability mass     : NOT YET VERIFIED")
print("Progress                  : 4 / 6 code cells")


XLKD-00 CELL 4 — PSEUDO-CAPTION ALIGNMENT
Rows / unique images      : 113287 / 113287
Teacher token occurrences : 1304640
Occurrence coverage       : 100.000000%
Length alignment          : 100.000000%
Position exact alignment  : 100.000000%
Unmapped observed tokens  : 0
Soft probability mass     : NOT YET VERIFIED
Progress                  : 4 / 6 code cells


## Cell 5 — Structural gate, next protocol và đề cương alignment


In [5]:
STARTED = time.perf_counter()
structural_gate = {
    "vocabulary_count_coverage_at_least_0_99": (
        vocabulary_count_coverage >= 0.99
    ),
    "mapping_collisions_zero": collision_count == 0,
    "special_tokens_4_of_4": len(special_map) == 4,
    "pseudo_occurrence_coverage_at_least_0_9999": (
        occurrence_coverage >= 0.9999
    ),
    "pseudo_length_alignment_at_least_0_9999": (
        length_alignment_rate >= 0.9999
    ),
    "pseudo_position_alignment_at_least_0_9999": (
        position_alignment_rate >= 0.9999
    ),
    "test_remains_sealed": (
        P1["test_set_status"] == "sealed"
        and P3B["test_set_status"] == "sealed"
        and P3E["test_set_status"] == "sealed"
    ),
}
structural_passed = all(structural_gate.values())
assert structural_passed, structural_gate

protocol = {
    "schema_version": "xlkd_cross_vocab_logit_kd_protocol_v1",
    "created_at_utc": now(),
    "status": "structural_preflight_passed",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method_name": "Cross-Vocabulary Mapped Logit KD",
    "direct_raw_logit_kd_allowed": False,
    "mapping_contract": {
        "relative_path": MAPPING_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(MAPPING_PATH),
    },
    "initial_hyperparameters": {
        "temperature": 4.0,
        "alpha_kd": 0.5,
        "alpha_human_ce": 0.5,
        "reason": (
            "Match the existing C→S3 Logit-KD contract and avoid an "
            "unregistered hyperparameter search."
        ),
    },
    "loss": (
        "(1-alpha)*CE_human + alpha*T^2*KL("
        "mapped_teacher_distribution || student_distribution)"
    ),
    "teacher_conditioning": (
        "Teacher-forcing with native IDs of the fixed pseudo-caption; "
        "Student uses the aligned Phase-1 IDs of the same word sequence."
    ),
    "next_stage": {
        "name": "XLKD-01 GPU mapped probability-mass preflight",
        "training_authorized": False,
        "sample_source": "Train+Restval only",
        "fixed_sample_size": 1024,
        "fixed_sample_rule": (
            "sort by SHA256(seed=42 || image_id), take first 1024"
        ),
        "temperature": 4.0,
        "top_k_candidates": [32, 64, 128],
        "required_measurements": [
            "mapped probability mass mean/p05/min",
            "top-k retained mass mean/p05/min",
            "unmapped mass mean/p95/max",
            "finite-logit rate",
            "position-mask correctness",
            "GPU peak memory and generation time",
        ],
        "go_gate": {
            "mean_mapped_probability_mass_min": 0.98,
            "p05_mapped_probability_mass_min": 0.95,
            "finite_logit_rate": 1.0,
            "position_mask_correctness": 1.0,
            "top_k_choice_rule": (
                "smallest k whose mean retained mapped mass >=0.98 "
                "and p05 >=0.95"
            ),
        },
    },
    "full_training_authorized": False,
    "validation_used_for_mapping_or_threshold_selection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
PROTOCOL_PATH = CONFIG / "xlkd_cross_vocab_logit_kd_protocol.json"
write_json(PROTOCOL_PATH, protocol)

alignment = {
    "schema_version": "xlkd_thesis_alignment_v1",
    "created_at_utc": now(),
    "status": "aligned_as_controlled_ablation",
    "overall_assessment": (
        "Phù hợp đề cương vì trực tiếp khảo sát một chiến lược KD cho "
        "image captioning, so sánh với Student baseline và Sequence KD, "
        "đồng thời phân tích chất lượng và chi phí tính toán."
    ),
    "supports_thesis_requirements": {
        "survey_kd_strategies": True,
        "apply_kd_to_image_captioning": True,
        "matched_teacher_student_experiment": True,
        "standard_validation_metrics": True,
        "three_seed_reproducibility_if_full_run_authorized": True,
        "quality_compute_tradeoff": True,
        "mobile_student_target_preserved": True,
    },
    "scientific_constraints": [
        "Report as cross-vocabulary mapped Logit KD, not direct Logit KD.",
        "Keep Sequence KD R1 as the official reference branch.",
        "Do not overwrite Phase 4/5/6 conclusions.",
        "Publish negative results if the branch fails.",
        "Use the same S1 baseline seeds 42/123/2026.",
        "Validation-only development; Test remains sealed.",
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
}
ALIGNMENT_PATH = REPORT / "xlkd00_thesis_alignment.json"
write_json(ALIGNMENT_PATH, alignment)
REPORT_MD = REPORT / "XLKD00_THESIS_ALIGNMENT.md"
REPORT_MD.write_text(
    """# Đánh giá phù hợp đề cương — T1→S1 Cross-Vocabulary Logit KD

## Kết luận

Nhánh thử nghiệm **phù hợp đề cương dưới dạng controlled ablation**. Nó trực
tiếp đáp ứng nội dung khảo sát chiến lược KD cho image captioning, huấn luyện
Student dưới sự hướng dẫn của Teacher, so sánh chất lượng và chi phí tính toán.

## Điều kiện khoa học

- Không gọi là direct Logit KD.
- Không ghi đè nhánh Sequence KD chính thức.
- Báo cáo cả kết quả âm.
- Giữ ba seed 42/123/2026 và cùng Student baseline.
- Không dùng Validation để học vocabulary map.
- Held-out Test luôn SEALED.
""",
    encoding="utf-8",
)
write_json(AUDIT / "xlkd00_cell05_decision.json", {
    "schema_version": "xlkd00_cell05_v1",
    "created_at_utc": now(),
    "status": "passed",
    "structural_gate": structural_gate,
    "structural_gate_passed": structural_passed,
    "direct_logit_kd_allowed": False,
    "mapped_probability_mass_probe_authorized": True,
    "full_training_authorized": False,
    "thesis_alignment": "aligned_as_controlled_ablation",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-00 CELL 5 — DECISION")
print("=" * 78)
print("Structural gate           : PASSED")
print("Raw direct Logit KD       : PROHIBITED")
print("Mapped mass GPU probe     : AUTHORIZED")
print("Full training             : NOT AUTHORIZED")
print("Thesis alignment          : ALIGNED AS CONTROLLED ABLATION")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 6 code cells")


XLKD-00 CELL 5 — DECISION
Structural gate           : PASSED
Raw direct Logit KD       : PROHIBITED
Mapped mass GPU probe     : AUTHORIZED
Full training             : NOT AUTHORIZED
Thesis alignment          : ALIGNED AS CONTROLLED ABLATION
Test Set status           : SEALED
Progress                  : 5 / 6 code cells


## Cell 6 — Package XLKD-00 và authorize XLKD-01


In [6]:
STARTED = time.perf_counter()
authorization = {
    "schema_version": "xlkd01_gpu_probability_mass_preflight_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "XLKD-01 ExpansionNet v2→S1 GPU mapped probability-mass "
        "and top-k retention preflight"
    ),
    "mapping_contract": {
        "relative_path": MAPPING_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(MAPPING_PATH),
    },
    "protocol": {
        "relative_path": PROTOCOL_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(PROTOCOL_PATH),
    },
    "temperature": 4.0,
    "fixed_sample_size": 1024,
    "top_k_candidates": [32, 64, 128],
    "teacher_checkpoint_sha256": P3B["checkpoint_sha256"],
    "full_training_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
NEXT_AUTH = OUT / "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
write_json(NEXT_AUTH, authorization)

tracked = (
    sorted(AUDIT.glob("*.json"))
    + sorted(CONFIG.glob("*.json"))
    + sorted(RESULT.glob("*"))
    + sorted(REPORT.glob("*"))
)
manifest_rows = []
for path in tracked:
    manifest_rows.append({
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
manifest_path = MANIFEST / "xlkd00_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
sums_path = MANIFEST / "SHA256SUMS"
sums_path.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

marker = {
    "schema_version": "xlkd00_expansionnet_s1_compatibility_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method": "Cross-Vocabulary Mapped Logit KD",
    "direct_logit_kd_allowed": False,
    "structural_mapping_gate": "passed",
    "mapped_teacher_tokens": mapped_count,
    "vocabulary_count_coverage": vocabulary_count_coverage,
    "pseudo_occurrence_coverage": occurrence_coverage,
    "pseudo_position_alignment_rate": position_alignment_rate,
    "soft_probability_mass_verified": False,
    "full_training_authorized": False,
    "xlkd01_authorized": True,
    "xlkd01_authorization_sha256": sha256(NEXT_AUTH),
    "thesis_alignment": "aligned_as_controlled_ablation",
    "manifest": {
        "csv": {
            "relative_path": manifest_path.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_path),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": sums_path.relative_to(OUT).as_posix(),
            "sha256": sha256(sums_path),
        },
    },
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
MARKER_PATH = OUT / "XLKD00_EXPANSIONNET_S1_COMPATIBILITY_COMPLETE.json"
write_json(MARKER_PATH, marker)

print("=" * 78)
print("XLKD-00 FINAL STATUS")
print("=" * 78)
print("Status                    : COMPLETE")
print("Vocabulary mapping        :", mapped_count, "/ 10000")
print("Vocabulary count coverage :", f"{vocabulary_count_coverage:.4%}")
print("Pseudo occurrence coverage:", f"{occurrence_coverage:.6%}")
print("Position alignment        :", f"{position_alignment_rate:.6%}")
print("Direct raw Logit KD       : PROHIBITED")
print("XLKD-01 GPU mass probe    : AUTHORIZED")
print("Full training             : NOT AUTHORIZED")
print("Thesis alignment          : YES, CONTROLLED ABLATION")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 6 code cells")


XLKD-00 FINAL STATUS
Status                    : COMPLETE
Vocabulary mapping        : 9959 / 10000
Vocabulary count coverage : 99.5900%
Pseudo occurrence coverage: 100.000000%
Position alignment        : 100.000000%
Direct raw Logit KD       : PROHIBITED
XLKD-01 GPU mass probe    : AUTHORIZED
Full training             : NOT AUTHORIZED
Thesis alignment          : YES, CONTROLLED ABLATION
Test Set status           : SEALED
Progress                  : 6 / 6 code cells
